In [1]:
import requests
import settings
import json
import os
import time
from typing import Dict, List, Optional, Any
from datetime import datetime

In [2]:
class Credential:
    _json_file = "models_data_final.json"  # Mismo archivo que Model
    _credentials_key = "credentials"  # Key específica para credenciales
    
    def __init__(self, definition_id, credential_info, owner_did: str = None):
        self.id = definition_id
        self.info = credential_info
        self.owner_did = owner_did
        self.timestamp = self._get_timestamp()
        self.credential_id = self._generate_credential_id()
        
        # Auto-saving al crear la instancia
        self._save_to_json()

    def _generate_credential_id(self):
        """Genera un ID único para la credencial"""
        import uuid
        return f"cred_{uuid.uuid4().hex[:8]}"

    def _get_timestamp(self):
        """Obtiene timestamp actual"""
        return datetime.now().isoformat()

    def _save_to_json(self):
        """Guarda la credencial en el archivo JSON"""
        try:
            # Cargar datos existentes
            existing_data = self._load_existing_data()
            
            # Preparar datos de la credencial
            credential_data = {
                "credential_id": self.credential_id,
                "definition_id": self.id,
                "info": self.info,
                "owner_did": self.owner_did,
                "timestamp": self.timestamp,
                "type": self.__class__.__name__
            }
            
            # Agregar datos específicos si existen
            if hasattr(self, '_get_credential_data'):
                credential_data.update(self._get_credential_data())
            
            # Actualizar o crear la sección de credenciales
            if self._credentials_key not in existing_data:
                existing_data[self._credentials_key] = []
            
            # Verificar si ya existe esta credencial (por ID)
            existing_index = None
            for i, cred in enumerate(existing_data[self._credentials_key]):
                if cred.get('credential_id') == self.credential_id:
                    existing_index = i
                    break
            
            if existing_index is not None:
                # Actualizar credencial existente
                existing_data[self._credentials_key][existing_index] = credential_data
            else:
                # Agregar nueva credencial
                existing_data[self._credentials_key].append(credential_data)
            
            # Guardar en archivo
            self._save_data_to_file(existing_data)
            
        except Exception as e:
            print(f"❌ Error guardando credencial en JSON: {e}")

    def _load_existing_data(self) -> Dict:
        """Carga los datos existentes del archivo JSON"""
        if os.path.exists(self._json_file):
            try:
                with open(self._json_file, 'r', encoding='utf-8') as f:
                    return json.load(f)
            except json.JSONDecodeError:
                return {}
        return {}

    def _save_data_to_file(self, data: Dict):
        """Guarda los datos en el archivo JSON"""
        with open(self._json_file, 'w', encoding='utf-8') as f:
            json.dump(data, f, indent=2, ensure_ascii=False)

    def update_info(self, new_info: Dict):
        """Actualiza la información de la credencial y guarda automáticamente"""
        self.info.update(new_info)
        self.timestamp = self._get_timestamp()
        self._save_to_json()

    def delete(self):
        """Elimina la credencial del JSON"""
        try:
            existing_data = self._load_existing_data()
            
            if self._credentials_key in existing_data:
                # Filtrar la credencial a eliminar
                existing_data[self._credentials_key] = [
                    cred for cred in existing_data[self._credentials_key] 
                    if cred.get('credential_id') != self.credential_id
                ]
                
                self._save_data_to_file(existing_data)
                print(f"✅ Credencial {self.credential_id} eliminada")
                
        except Exception as e:
            print(f"❌ Error eliminando credencial: {e}")

    @classmethod
    def load_all_credentials(cls) -> List[Dict]:
        """Carga todas las credenciales del archivo JSON"""
        try:
            existing_data = cls._load_existing_data(cls)
            return existing_data.get(cls._credentials_key, [])
        except Exception as e:
            print(f"❌ Error cargando credenciales: {e}")
            return []

    @classmethod
    def find_by_owner_did(cls, did: str) -> List[Dict]:
        """Encuentra credenciales por DID del propietario"""
        all_credentials = cls.load_all_credentials()
        return [cred for cred in all_credentials if cred.get('owner_did') == did]

    @classmethod
    def find_by_definition_id(cls, definition_id: str) -> List[Dict]:
        """Encuentra credenciales por ID de definición"""
        all_credentials = cls.load_all_credentials()
        return [cred for cred in all_credentials if cred.get('definition_id') == definition_id]

    def to_dict(self) -> Dict:
        """Convierte la credencial a diccionario"""
        return {
            "credential_id": self.credential_id,
            "definition_id": self.id,
            "info": self.info,
            "owner_did": self.owner_did,
            "timestamp": self.timestamp,
            "type": self.__class__.__name__
        }

In [3]:
class ACApyClient:
    def __init__(self, wallet_token: Optional[str] = None):
        self.admin_url = settings.ACA_PY_CONFIG['admin_url']
        self.seeder = "V4SGRU86Z58d6TV7PBUe6f"
        self.headers = {
            'Content-Type': 'application/json',
            'Accept': 'application/json',
        }
        if wallet_token:
            self.headers['Authorization'] = f'Bearer {wallet_token}'

    def create_wallet(self, wallet_name: str, wallet_key: str, label: str = None):
        """Crea un nuevo wallet en modo multitenant"""
        url = f"{self.admin_url}/multitenancy/wallet"
        payload = {
            "wallet_name": wallet_name,
            "wallet_key": wallet_key,
            "label": label or wallet_name,
            "wallet_type": "askar",
            "key_management_mode": "managed",
            "seed": self.seeder
        }
        
        response = requests.post(url, json=payload, headers=self.headers)
        response.raise_for_status()
        return response.json()

    def get_wallet_token(self, wallet_id: str, wallet_key: str):
        """Obtiene token de acceso para un wallet específico"""
        url = f"{self.admin_url}/multitenancy/wallet/{wallet_id}/token"
        payload = {
            "wallet_key": wallet_key
        }
        response = requests.post(url, json=payload, headers=self.headers)
        response.raise_for_status()
        return response.json()['token']

    def delete_wallet(self, wallet_id: str):
        """Elimina un wallet multitenant"""
        url = f"{self.admin_url}/multitenancy/wallet/{wallet_id}"
        response = requests.delete(url, headers=self.headers)
        response.raise_for_status()
        return response.json()

    def create_local_did(self):
        """Crea un DID local dentro del wallet del agente"""
        url = f"{self.admin_url}/wallet/did/create"
        payload = {"method": "sov", "options": {"key_type": "ed25519"}}
        response = requests.post(url, json=payload, headers=self.headers)
        response.raise_for_status()
        return response.json().get("result", {})

    def register_did_in_ledger(self, did, verkey):
        """Registrar el DID en el ledger de Indy"""
        
        # PRIMERO: Configurar el DID como público en el wallet del tenant
        set_public_url = f"{self.admin_url}/wallet/did/public"
        set_public_payload = {
            "did": did
        }
        
        print(f"📤 Configurando DID como público: {did}")
        public_response = requests.post(set_public_url, params=set_public_payload, headers=self.headers)
        
        print(f"📥 Response configurar público: {public_response.status_code}")
        if public_response.status_code != 200:
            print(f"❌ Error configurando DID público: {public_response.text}")
            # Continuar de todas formas, a veces igual funciona
        
        # LUEGO: Registrar en el ledger
        url = f"{self.admin_url}/ledger/register-nym"
        payload = {
            "did": did,
            "verkey": verkey,
            "alias": "AriesCLI",
            "role": "ENDORSER"
        }

        print(f"📤 Enviando request a: {url}")
        print(f"📝 Payload: {json.dumps(payload, indent=2)}")

        # CORRECCIÓN: Usar json= en lugar de params=
        response = requests.post(url, params=payload, headers=self.headers)
        
        print(f"📥 Status: {response.status_code}")
        print(f"📥 Body: {response.text}")

        if response.ok:
            try:
                return response.json()
            except json.JSONDecodeError:
                return {"raw_response": response.text}
        else:
            return {
                "error": f"HTTP {response.status_code}",
                "raw_response": response.text
            }

    def register_wallet(self):
        """Crea un DID local y lo registra en el ledger"""
        did_info = self.create_local_did()
        did = did_info["did"]
        verkey = did_info["verkey"]
        ledger_response = self.register_did_in_ledger(did, verkey)
        return {"did": did, "verkey": verkey, "ledger_tx": ledger_response}

    # ------------------------
    # 📜 Schemas y CredDefs
    # ------------------------
    def register_credential_schema(self, name, version, attributes):
        """Registra un nuevo schema en el ledger"""
        schema_payload = {
            "schema_name": name,
            "schema_version": version,
            "attributes": attributes
        }
        url = f"{self.admin_url}/schemas"
        response = requests.post(url, json=schema_payload, headers=self.headers)
        response.raise_for_status()
        result = response.json()
        return result.get("schema", result.get("sent", {}))

    def create_credential_definition(self, schema_id, tag="default", support_revocation=False):
        """Crea una credential definition basada en un schema existente"""
        url = f"{self.admin_url}/credential-definitions"
        payload = {
            "schema_id": schema_id,
            "tag": tag,
            "support_revocation": support_revocation
        }
        response = requests.post(url, json=payload, headers=self.headers)
        response.raise_for_status()
        data = response.json()
        return data.get("credential_definition_id")

    def create_credential(self, name, version, attributes, support_revocation=False):        
        try:
            # Registrar schema
            schema_response = self.register_credential_schema(name, version, attributes)
            
            if schema_response:
                # Crear credential definition
                cred_def_id = self.create_credential_definition(
                    schema_id=schema_response['id'],
                    support_revocation=support_revocation
                )
                return Credential(cred_def_id, schema_response)
            return None
            
        except Exception as e:
            print(f"❌ Error creando credencial: {e}")
            return None

    def get_existing_schemas(self):
        """Obtiene todos los schemas existentes del ledger"""
        try:
            url = f"{self.admin_url}/schemas/created"
            response = requests.get(url, headers=self.headers)
            response.raise_for_status()
            return response.json().get('schema_ids', [])
        except Exception as e:
            print(f"⚠️ Error obteniendo schemas existentes: {e}")
            return []
    
    def schema_exists(self, schema_name: str, schema_version: str) -> bool:
        """Verifica rápidamente si un schema ya existe"""
        existing_schemas = self.get_existing_schemas()
        
        # Buscar en la lista de schema_ids
        target_pattern = f"{schema_name}/{schema_version}"
        for schema_id in existing_schemas:
            if target_pattern in schema_id:
                return True
        return False
    
    def get_existing_cred_defs(self):
        """Obtiene todas las credential definitions existentes"""
        try:
            url = f"{self.admin_url}/credential-definitions/created"
            response = requests.get(url, headers=self.headers)
            response.raise_for_status()
            return response.json().get('credential_definition_ids', [])
        except Exception as e:
            print(f"⚠️ Error obteniendo cred defs existentes: {e}")
            return []
    
    def get_cred_def_by_schema(self, schema_id: str):
        """Obtiene credential definition por schema_id"""
        cred_defs = self.get_existing_cred_defs()
        print(f"🔍 Cred Defs disponibles: {cred_defs}")

        for cred_def_id in cred_defs:
            if schema_id in cred_def_id:
                print(f"✅ Cred Def encontrada: {cred_def_id}")
                return cred_def_id
            
        print(f"❌ No se encontró Cred Def para schema: {schema_id}")
        return None
    
    # ------------------------
    # 🔗 Conexiones
    # ------------------------
    def create_invitation(self):
        """Crea una invitación de conexión"""
        url = f"{self.admin_url}/connections/create-invitation"
        response = requests.post(url, headers=self.headers)
        response.raise_for_status()
        return response.json()

    def accept_invitation(self, invitation_url: str):
        """Acepta una invitación de conexión"""
        url = f"{self.admin_url}/connections/receive-invitation"
        # Extraer el payload de la URL de invitación
        invitation_payload = self._parse_invitation_url(invitation_url)
        response = requests.post(url, json=invitation_payload, headers=self.headers)
        response.raise_for_status()
        return response.json()

    def _parse_invitation_url(self, invitation_url: str):
        """Parsea una URL de invitación en payload JSON"""
        # Implementar parsing de URL de invitación
        # Esto es un ejemplo simplificado
        import base64
        import json as json_lib
        
        if invitation_url.startswith('http'):
            # Es una URL completa, extraer el parámetro de invitación
            from urllib.parse import urlparse, parse_qs
            parsed = urlparse(invitation_url)
            query_params = parse_qs(parsed.query)
            if 'c_i' in query_params:
                invitation_b64 = query_params['c_i'][0]
                invitation_json = base64.urlsafe_b64decode(invitation_b64 + '==')
                return json_lib.loads(invitation_json)
        
        # Si ya es JSON, devolver como está
        try:
            return json_lib.loads(invitation_url)
        except:
            raise ValueError("Formato de invitación no válido")

    # ------------------------
    # 🎓 Emisión de Credenciales
    # ------------------------
    def send_credential_offer(self, cred_def_id, attributes):
        """
        Envía una oferta de credencial usando una cred_def existente.
        attributes: lista de dicts [{"name": "campo", "value": "valor"}]
        """
        invitation = self.create_invitation()
        connection_id = invitation["connection_id"]

        offer_payload = {
            "connection_id": connection_id,
            "cred_def_id": cred_def_id,
            "credential_preview": {
                "@type": "issue-credential/1.0/credential-preview",
                "attributes": attributes
            },
            "auto_issue": True,
            "auto_remove": True
        }

        print(f"📤 Enviando oferta de credencial: {json.dumps(offer_payload, indent=2)}")

        url = f"{self.admin_url}/issue-credential/send-offer"
        response = requests.post(url, json=offer_payload, headers=self.headers)
        print(f"📥 Status: {response.status_code}")
        print(f"📥 Body: {response.text}")

        if response.ok:
            return response.json()
        else:
            return {
                "error": f"HTTP {response.status_code}",
                "raw_response": response.text
            }

    def get_credential_offers(self):
        """Obtiene todas las ofertas de credenciales pendientes"""
        url = f"{self.admin_url}/issue-credential/records"
        response = requests.get(url, headers=self.headers)
        response.raise_for_status()
        return response.json().get('results', [])

    def accept_credential_offer(self, cred_ex_id: str):
        """Acepta una oferta de credencial"""
        url = f"{self.admin_url}/issue-credential/records/{cred_ex_id}/send-request"
        response = requests.post(url, headers=self.headers)
        response.raise_for_status()
        return response.json()

    def store_credential(self, cred_ex_id: str):
        """Almacena una credencial recibida"""
        url = f"{self.admin_url}/issue-credential/records/{cred_ex_id}/store"
        response = requests.post(url, headers=self.headers)
        response.raise_for_status()
        return response.json()

    def get_credentials(self):
        """Obtiene todas las credenciales almacenadas"""
        url = f"{self.admin_url}/credentials"
        response = requests.get(url, headers=self.headers)
        response.raise_for_status()
        return response.json().get('results', [])

    def get_credential_by_id(self, credential_id: str):
        """Obtiene una credencial específica por ID"""
        url = f"{self.admin_url}/credentials/{credential_id}"
        response = requests.get(url, headers=self.headers)
        response.raise_for_status()
        return response.json()

In [4]:
class ACApyClientTenant:
    def __init__(self, wallet_token: str):
        self.admin_url = settings.ACA_PY_CONFIG['admin_url']
        self.seeder = "V4SGRU86Z58d6TV7PBUe6f"
        self.headers = {
            'Content-Type': 'application/json',
            'Accept': 'application/json',
            'Authorization': f'Bearer {wallet_token}',
        }
        self.base_client = ACApyClient()
        self.did = None
        self.verkey = None

    def register_wallet(self):
        public_did = self.base_client.create_local_did()
        response = self.register_did_in_ledger(public_did['did'], public_did['verkey'])
        print(response)
        return response

    def register_did_in_ledger(self, did, verkey):
        """Registrar el DID en el ledger usando el agente base"""
        url = f"{self.base_client.admin_url}/ledger/register-nym"
        params = {
            "did": did,
            "verkey": verkey,
            "alias": "AriesCLI",
            "role": "ENDORSER"
        }
        response = requests.post(url, params=params, headers=self.base_client.headers)
        if response.ok:
            self.did = did
            self.verkey= verkey
            return response.json()
        else:
            return {"error": response.text, "status": response.status_code}
    
    def get_credentials(self):
        """
        Obtiene todas las credenciales almacenadas en el wallet del tenant.
        """
        try:
            url = f"{self.admin_url}/credentials"
            response = requests.get(url, headers=self.headers)
            response.raise_for_status()
            
            data = response.json()
            credentials = data.get('results', [])
            
            print(f"✅ {len(credentials)} credenciales obtenidas del wallet")
            return credentials

        except Exception as e:
            print(f"❌ Error obteniendo credenciales: {e}")
            return []
    
    def get_credential_offers(self):
        """
        Obtiene todas las ofertas de credenciales pendientes (state=offer-received)
        desde el agente Aries.
        """
        url = f"{self.admin_url}/issue-credential-2.0/records?state=offer-received"

        response = requests.get(url, headers=self.headers)

        if response.status_code != 200:
            raise Exception(f"Error al obtener ofertas de credenciales: {response.text}")

        data = response.json()
        return data.get("results", [])

In [5]:
class Wallet:
    def __init__(self, wallet_name: Optional[str] = None):
        self.wallet_name = wallet_name or f"wallet_{self._generate_id()}"
        self.wallet_key = self._generate_wallet_key()
        self.wallet_id = None
        self.wallet_token = None
        self.client = None
        self.wallet_data = {}
        
        self._initialize_wallet()

    def _initialize_wallet(self):
        """Inicializa el wallet en ACA-Py"""
        try:
            # Crear cliente base (sin token)
            base_client = ACApyClient()
            
            # Crear wallet multitenant
            wallet_info = base_client.create_wallet(
                wallet_name=self.wallet_name,
                wallet_key=self.wallet_key,
                label=self.wallet_name
            )
            
            self.wallet_id = wallet_info['wallet_id']
            
            # Obtener token de acceso
            self.wallet_token = base_client.get_wallet_token(
                self.wallet_id, 
                self.wallet_key
            )
            
            # Crear cliente autenticado
            self.client = ACApyClientTenant(self.wallet_token)
            
            # Registrar DID
            wallet_data = self.client.register_wallet()
            self.wallet_data = wallet_data
            self.wallet_data['wallet_id'] = self.wallet_id
            self.wallet_data['wallet_name'] = self.wallet_name
            
        except Exception as e:
            print(f"❌ Error inicializando wallet {self.wallet_name}: {e}")
            raise

    def _generate_wallet_key(self):
        """Generar clave segura para el wallet"""
        import secrets
        import string
        alphabet = string.ascii_letters + string.digits
        return ''.join(secrets.choice(alphabet) for _ in range(32))

    def _generate_id(self):
        """Generar ID único"""
        import uuid
        return str(uuid.uuid4())[:8]

    def accept_credential_invitation(self, invitation_url: str):
        """Acepta una invitación de credencial"""
        return self.client.accept_invitation(invitation_url)

    def get_pending_credential_offers(self):
        """Obtiene ofertas de credenciales pendientes"""
        return self.client.get_credential_offers()

    def accept_all_pending_credentials(self):
        """Acepta y almacena todas las credenciales pendientes"""
        offers = self.get_pending_credential_offers()
        stored_credentials = []
        
        for offer in offers:
            if offer['state'] == 'offer_received':
                try:
                    # Aceptar oferta
                    cred_ex_id = offer['cred_ex_id']
                    self.client.accept_credential_offer(cred_ex_id)
                    
                    # Almacenar credencial
                    stored_cred = self.client.store_credential(cred_ex_id)
                    stored_credentials.append(stored_cred)
                    
                except Exception as e:
                    print(f"❌ Error aceptando credencial {offer['cred_ex_id']}: {e}")
        
        return stored_credentials

    def get_stored_credentials(self):
        """Obtiene todas las credenciales almacenadas"""
        return self.client.get_credentials()

In [74]:
evtol_credential = {
    "name":"Evtol_Crede",
    "version": "11.0",
    "schema": ["id_puerto", "updates", "status"]
    }
user_credential = {
    "name":"User_Cred",
    "version": "11.0",
    "schema": ["first_name", "last_name", "can_ride"]
}
vertiport_credential = {
    "name":"Port_Credential",
    "version": "11.0",
    "schema": ["last_name", "n_airstrip", "n_parkings", "n_free_airstrip", "n_free_parkings", "coord_lon","coord_lat"]
}

credential_definitions = {
    "evtol": evtol_credential,
    "user": user_credential,
    "vertiport": vertiport_credential
}

In [79]:
import json
import os

class Issuer:

    def __init__(self, credential_configs, json_filename="issuer_credentials.json"):
        """
        Inicializa el cliente y procesa la creación de esquemas y definiciones.
        Si el archivo JSON ya existe, carga los datos y evita recrear credenciales.
        """
        self.client = ACApyClient()
        self.credential_configs = credential_configs
        self.json_filename = json_filename
        self.invitation_url = f"{self.client.admin_url}/connections/create-invitation"

        self.admin_url = self.client.admin_url

        # Verificar si ya existe el archivo de credenciales
        if os.path.exists(self.json_filename):
            print(f"📂 Archivo {self.json_filename} encontrado. Cargando credenciales existentes...")
            self._load_from_json()
        else:
            print(f"⚠️ {self.json_filename} no existe. Creando credenciales por primera vez...")
            self.issuer_data = {
                "schemas": {},
                "credential_definitions": {}
            }
            self._create_all_credentials()
            self._save_to_json()

    def get_admin_url(self):
        return self.admin_url

    def get_invitation_url(self):
        return self.invitation_url
    
    def get_headers(self):
        return self.client.headers

    def _create_all_credentials(self):
        """Crea los esquemas y credential definitions para cada credencial definida por el usuario."""
        for cred_key, cred_info in self.credential_configs.items():

            print(f"📌 Registrando schema para: {cred_info['name']}...")

            # Crear schema
            schema = self.client.register_credential_schema(
                name=cred_info["name"],
                version=cred_info["version"],
                attributes=cred_info["schema"]
            )

            # Guardar schema generado
            self.issuer_data["schemas"][cred_key] = schema

            print(f"📌 Creando credential definition para schema ID: {schema['id']}")

            # Crear credential definition
            cred_def = self.client.create_credential_definition(
                schema_id=schema["id"],
                support_revocation=False
            )

            # Guardar definición generada
            self.issuer_data["credential_definitions"][cred_key] = cred_def


    def _save_to_json(self):
        """Guarda la información del issuer en un JSON local."""
        with open(self.json_filename, "w") as f:
            json.dump(self.issuer_data, f, indent=4)
        print(f"✅ Información guardada en {self.json_filename}")


    def _load_from_json(self):
        """Carga la información de un archivo JSON existente."""
        with open(self.json_filename, "r") as f:
            self.issuer_data = json.load(f)
        print("✅ Credenciales cargadas correctamente desde el archivo.")


    def get_data(self):
        """Retorna los datos almacenados en memoria."""
        return self.issuer_data

    def get_entity_credential(self, entity):
        try:        
            return {
                "schema_info": self.issuer_data['schemas'][entity],
                "definition": self.issuer_data['credential_definitions'][entity]
            }
        except:
            return {}


In [76]:
issuer = Issuer(credential_definitions)

data = issuer.get_data()
print(data)

⚠️ issuer_credentials.json no existe. Creando credenciales por primera vez...
📌 Registrando schema para: Evtol_Crede...
📌 Creando credential definition para schema ID: V4SGRU86Z58d6TV7PBUe6f:2:Evtol_Crede:11.0
📌 Registrando schema para: User_Cred...
📌 Creando credential definition para schema ID: V4SGRU86Z58d6TV7PBUe6f:2:User_Cred:11.0
📌 Registrando schema para: Port_Credential...
📌 Creando credential definition para schema ID: V4SGRU86Z58d6TV7PBUe6f:2:Port_Credential:11.0
✅ Información guardada en issuer_credentials.json
{'schemas': {'evtol': {'ver': '1.0', 'id': 'V4SGRU86Z58d6TV7PBUe6f:2:Evtol_Crede:11.0', 'name': 'Evtol_Crede', 'version': '11.0', 'attrNames': ['status', 'updates', 'id_puerto'], 'seqNo': 59}, 'user': {'ver': '1.0', 'id': 'V4SGRU86Z58d6TV7PBUe6f:2:User_Cred:11.0', 'name': 'User_Cred', 'version': '11.0', 'attrNames': ['first_name', 'can_ride', 'last_name'], 'seqNo': 61}, 'vertiport': {'ver': '1.0', 'id': 'V4SGRU86Z58d6TV7PBUe6f:2:Port_Credential:11.0', 'name': 'Port_Cr

In [80]:
issuer.get_entity_credential('vertiport')

{'schema_info': {'ver': '1.0',
  'id': 'V4SGRU86Z58d6TV7PBUe6f:2:Port_Credential:11.0',
  'name': 'Port_Credential',
  'version': '11.0',
  'attrNames': ['n_free_airstrip',
   'last_name',
   'n_airstrip',
   'coord_lon',
   'coord_lat',
   'n_free_parkings',
   'n_parkings'],
  'seqNo': 63},
 'definition': 'V4SGRU86Z58d6TV7PBUe6f:3:CL:63:default'}

In [84]:
class Model:
    _instances: Dict[str, List[Dict[str, Any]]] = {}
    _json_file = "models_data.json"
    
    def __init__(self, id=None, wallet_name: Optional[str] = None, issuer:Issuer=None):
        self.id = id if id is not None else self._generate_id()
        self.wallet = Wallet(wallet_name)
        self.did = self.wallet.client.did
        self.verkey = self.wallet.client.verkey
        self.token = self.wallet.wallet_token
        self.credentials = []
        self.entity_type = self.__class__.__name__.lower()
        
        # Registrar instancia
        self._register_instance()
        
        # Crear credencial automáticamente
        self._create_entity_credential(issuer=issuer)
        
        # Cargar credenciales existentes
        self._load_credentials()
        
        # Aceptar cualquier credencial pendiente automáticamente
        self._accept_pending_credentials()

    def _create_entity_credential(self, issuer:Issuer):
        """Crea automáticamente la credencial correspondiente al tipo de entidad"""

        client_wallet = self.wallet
        print("DID ya registrado y funcional para el Indy : ", client_wallet.client.did)
        print("Verkey del DID: ", client_wallet.client.verkey)

        temp_entity_type = self.entity_type
        credential = issuer.get_entity_credential(temp_entity_type)
        print("Credencial de la entidad: ", credential)
        temp_cred_def_id = credential["definition"]
        temp_schema_id = credential["schema_info"]["id"]
        print("Credential definition a usar: ", temp_cred_def_id)
        print("Schema id a usar: ", temp_schema_id)

        temp_attributes = self._get_credential_attributes()
        temp_conn_data = {
            "their_label": client_wallet.wallet_name,
            "their_role": "holder"
        }

        print("Info del holder: ", temp_conn_data)
        # Apunta al endpoint del agente emisor
        temp_conn_url = f"{issuer.get_admin_url()}/connections/create-invitation"
        print("Endpoint del agente emisor: ", temp_conn_url)
        # Se hace un POST con la info básica del holder
        temp_conn_response = requests.post(temp_conn_url, json=temp_conn_data, headers=issuer.get_headers())
        temp_conn_response.raise_for_status()
        temp_connection = temp_conn_response.json()
        print("Conexion: ", temp_connection)
        # connection_id es el identificador local de la conexión
        temp_connection_id = temp_connection["connection_id"]
        print("Identificador local de la conexion: ", temp_connection_id)

        # HOLDERS
        temp_holder_client = client_wallet.client
        temp_receive_url = f"{temp_holder_client.admin_url}/connections/receive-invitation"
        print("Endpoint del holder que recibe la invitación: ", temp_receive_url)
        temp_holder_resp = requests.post(
            temp_receive_url,
            json=temp_connection["invitation"],
            headers=temp_holder_client.headers
        )
        print(temp_holder_resp.json())
        temp_holder_conn_id = temp_holder_resp.json()["connection_id"]
        print("Connection ID del holder:", temp_holder_conn_id)

        # Aquí podemos ver que el issuer ya recibió el connection request del holder y envió la connection response
        # Esto es debido a que el agente esta configurado como aceept auto, esto se puede verificar en:
        # La línea 53 y 54 de /SSI_App/aries_client/docker-compose.yml

        # ISSUER
        temp_issuer_check_url = f"{issuer.get_admin_url()}/connections/{temp_connection_id}"
        print("URL de la aceptación de solicitud del issuer: ", temp_issuer_check_url)
        print(requests.get(temp_issuer_check_url, headers=issuer.get_headers()).json())

        # HOLDER
        # En este caso la conexión debería verse completed, pero no se ve porque se está usando connections/1.0
        # donde el estado final es response
        temp_holder_check_url = f"{temp_holder_client.admin_url}/connections/{temp_holder_conn_id}"
        print("URL del estado del holder:", temp_holder_check_url)
        print(requests.get(temp_holder_check_url, headers=temp_holder_client.headers).json())
        
        # CONSTRUCCIÓN DE LA CREDENCIAL
        temp_credential_data = {
            "connection_id": temp_connection_id,
            "credential_preview": {
                # Esta sección del mensaje DIDComm sirve para mostrar o anunciar los atributos del credencial antes de emitirlo formalmente
                "@type": "https://didcomm.org/issue-credential/2.0/credential-preview",
                "attributes": temp_attributes
            },
            "filter": {
                "indy": {
                    "schema_id": temp_schema_id,
                    "cred_def_id": temp_cred_def_id
                }
            },
            "auto_remove": False,
            "trace": False
        }
        print("Schema: ", temp_schema_id)
        print("Definition: ", temp_cred_def_id)


        requests.get(f"{issuer.get_admin_url()}/connections/{temp_connection_id}", headers=issuer.get_headers()).json()

        # PRUEBA DE EMISIÓN (POR FAVOR DIOSITO)
        # ISSUER
        temp_issuer_cred_url = f"{issuer.get_admin_url()}/issue-credential-2.0/send"

        # Envía una solicitud HTTP POST al endpoint con todos los datos de la credencial
        temp_issuer_cred_response = requests.post(
            temp_issuer_cred_url,
            json=temp_credential_data,
            headers=issuer.get_headers()
        )

        print("Status:", temp_issuer_cred_response.status_code)
        print("\n\nResultado temp_issuer_cred_response:",temp_issuer_cred_response.json())

        holder_offers = requests.get(
            f"{temp_holder_client.admin_url}/issue-credential-2.0/records",
            headers=temp_holder_client.headers
        ).json()
        print("Holder offers: ", holder_offers)
        # Obtenemos el cred_ex_id del holder y verificamos que es diferente al del issuer
        temp_issuer_cred_json = temp_issuer_cred_response.json()
        temp_issuer_cred_ex_id = temp_issuer_cred_json["cred_ex_id"]
        temp_holder_cred_ex_id = None

        for rec in holder_offers["results"]:
            # Si la clave esta dentro del subregistro 'cred_ex_record'
            if "cred_ex_record" in rec:
                record = rec["cred_ex_record"]
            else:
                record = rec

            if "state" in record and record["state"] == "offer-received":
                temp_holder_cred_ex_id = record["cred_ex_id"]
                break

        print("Holder cred_ex_id:", temp_holder_cred_ex_id)
        print("Issuer cred_ex_id:", temp_issuer_cred_ex_id)

        # HOLDER
        # Logramos revisar que el holder recibio la oferta ahora toca aceptarla
        temp_holder_request_url = f"{temp_holder_client.admin_url}/issue-credential-2.0/records/{temp_holder_cred_ex_id}/send-request"
        print("URL para aceptar la oferta:", temp_holder_request_url)
        temp_holder_request = requests.post(
            temp_holder_request_url,
            headers=temp_holder_client.headers
        )

        print("RAW RESPONSE:")
        print(temp_holder_request.text)

        # ISSUER
        # Ya que el estado es request-sent ahora toca que el issuer responda generando la credencial y enviandola
        temp_issuer_issue_url = f"{issuer.get_admin_url()}/issue-credential-2.0/records/{temp_issuer_cred_ex_id}/issue"
        print("URL para que el issuer responda:", temp_issuer_issue_url)
        temp_issuer_response = requests.post(
            temp_issuer_issue_url,
            headers=issuer.get_headers()
        )

        print("RAW RESPONSE:")
        print(temp_issuer_response.text)

        # Sin embargo esto da error, porque nuestro issuer esta configurado como auto_issue = true por lo que automáticamente emitio la credencial

        # Verificar el estado del issuer, podemos ver que ya esta en credential_issued
        if temp_issuer_cred_ex_id:
            check_url = f"{issuer.get_admin_url()}/issue-credential-2.0/records/{temp_issuer_cred_ex_id}"
            check_response = requests.get(check_url, headers=issuer.get_headers())
            print("Estado actual del credential exchange:")
            print(json.dumps(check_response.json(), indent=2))

        # Verificar estado del holder
        holder_check_url = f"{temp_holder_client.admin_url}/issue-credential-2.0/records/{temp_holder_cred_ex_id}"
        holder_check_response = requests.get(holder_check_url, headers=temp_holder_client.headers)
        print("Estado actual del holder:")
        print(json.dumps(holder_check_response.json(), indent=2))

        # HOLDER
        # Almacenar la credencial en el wallet y vemos que esta en estado done
        temp_holder_store_url = f"{temp_holder_client.admin_url}/issue-credential-2.0/records/{temp_holder_cred_ex_id}/store"
        print("URL para almacenar la credencial:", temp_holder_store_url)

        temp_holder_store_response = requests.post(
            temp_holder_store_url,
            headers=temp_holder_client.headers
        )

        print("Response status:", temp_holder_store_response.status_code)
        print("RAW RESPONSE:")
        print(temp_holder_store_response.text)

        self.credentials['auth'] = temp_holder_store_response['cred_ex_record']

    def _get_credential_attributes(self):
        """Obtiene los atributos específicos para la credencial - DEBE SER IMPLEMENTADO POR SUBCLASES"""
        # Este método debe ser sobrescrito por cada subclase
        return []

    def _generate_id(self):
        """Generar ID único para el modelo"""
        import uuid
        return str(uuid.uuid4())

    def _register_instance(self):
        """Registra la instancia en el JSON"""
        class_name = self.__class__.__name__
        if class_name not in Model._instances:
            Model._instances[class_name] = []
        
        instance_data = {
            "id": self.id,
            "did": self.did,
            "verkey": self.verkey,
            "token": self.token,
            "wallet_data": self.wallet.wallet_data,
            "credentials": self.credentials,
            "wallet_name": self.wallet.wallet_name,
            "timestamp": self._get_timestamp()
        }
        
        # Agregar datos específicos de la clase hija
        if hasattr(self, '_get_instance_data'):
            instance_data.update(self._get_instance_data())
        
        Model._instances[class_name].append(instance_data)
        self._save_to_json()

    def _save_to_json(self):
        """Guarda todas las instancias en JSON"""
        import json as json_lib
        import os
        
        try:
            existing_data = {}
            if os.path.exists(Model._json_file):
                with open(Model._json_file, 'r', encoding='utf-8') as f:
                    existing_data = json_lib.load(f)
            
            # Actualizar con datos actuales
            for class_name, instances in Model._instances.items():
                existing_data[class_name] = instances
            
            with open(Model._json_file, 'w', encoding='utf-8') as f:
                json_lib.dump(existing_data, f, indent=2, ensure_ascii=False)
                
        except Exception as e:
            print(f"❌ Error guardando en JSON: {e}")

    def _get_timestamp(self):
        """Obtiene timestamp actual"""
        from datetime import datetime
        return datetime.now().isoformat()

    def _load_credentials(self):
        """Carga credenciales almacenadas del wallet"""
        try:
            stored_creds = self.wallet.get_stored_credentials()
            self.credentials = stored_creds
        except Exception as e:
            print(f"⚠️ Error cargando credenciales: {e}")
            self.credentials = []

    def _accept_pending_credentials(self):
        """Acepta automáticamente credenciales pendientes"""
        try:
            new_credentials = self.wallet.accept_all_pending_credentials()
            if new_credentials:
                print(f"✅ {self.__class__.__name__} {self.id} aceptó {len(new_credentials)} credenciales")
                self._load_credentials()  # Recargar lista de credenciales
                self._register_instance()  # Actualizar en JSON
        except Exception as e:
            print(f"⚠️ Error aceptando credenciales pendientes: {e}")

    def accept_credential_invitation(self, invitation_url: str):
        """Acepta una invitación de credencial"""
        result = self.wallet.accept_credential_invitation(invitation_url)
        # Procesar credenciales pendientes después de aceptar invitación
        self._accept_pending_credentials()
        return result

    def get_credentials_info(self):
        """Obtiene información de las credenciales"""
        return {
            "total_credentials": len(self.credentials),
            "credentials": self.credentials
        }

In [85]:
# A partir de Model se puede crear la entidad de tipo Evtol
class Evtol(Model):
    def __init__(self, id=None, wallet_name=None, model="Standard", max_speed=120, id_puerto="default", status="active", issuer:Issuer=None):
        self.model = model
        self.max_speed = max_speed
        self.id_puerto = id_puerto
        self.status = status
        super().__init__(id, wallet_name,issuer=issuer)
    
    def _get_instance_data(self):
        return {
            "model": self.model,
            "max_speed": self.max_speed,
            "id_puerto": self.id_puerto,
            "status": self.status,
            "type": "eVTOL"
        }

    def _get_credential_attributes(self):
        """Implementa los atributos específicos para la credencial de Evtol"""
        return [
            {"name": "id_puerto", "value": self.id_puerto},
            {"name": "updates", "value": "0"},  # Inicialmente 0 updates
            {"name": "status", "value": self.status}
        ]

In [86]:
evtol_test = Evtol(id="VertiPort_0001", issuer=issuer)

{'success': True}
DID ya registrado y funcional para el Indy :  39zyrur6sE1m6PdXzvHfwi
Verkey del DID:  2B8VAwx1ifLnARZV8p8eGLvUWt2SouebPhcCQzUstV16
Credencial de la entidad:  {'schema_info': {'ver': '1.0', 'id': 'V4SGRU86Z58d6TV7PBUe6f:2:Evtol_Crede:11.0', 'name': 'Evtol_Crede', 'version': '11.0', 'attrNames': ['status', 'updates', 'id_puerto'], 'seqNo': 59}, 'definition': 'V4SGRU86Z58d6TV7PBUe6f:3:CL:59:default'}
Credential definition a usar:  V4SGRU86Z58d6TV7PBUe6f:3:CL:59:default
Schema id a usar:  V4SGRU86Z58d6TV7PBUe6f:2:Evtol_Crede:11.0
Info del holder:  {'their_label': 'wallet_29b570e9', 'their_role': 'holder'}
Endpoint del agente emisor:  http://localhost:9031/connections/create-invitation
Conexion:  {'connection_id': '50add463-d5db-4fab-9483-cc6aa4fe35da', 'invitation': {'@type': 'https://didcomm.org/connections/1.0/invitation', '@id': '19e58620-1f21-40f3-b6c8-1eaa4ede102e', 'label': 'Agent with Local Genesis', 'recipientKeys': ['Hms2bBSR7yPbCoyx4bJdxjXHUdCHbnAzma6MQRJtbxdj']

JSONDecodeError: Extra data: line 1 column 4 (char 3)

In [12]:
class User(Model):
    def __init__(self, id=None, wallet_name=None, first_name="", last_name="", can_ride=True):
        self.first_name = first_name
        self.last_name = last_name
        self.can_ride = can_ride
        super().__init__(id, wallet_name)
    
    def _get_instance_data(self):
        return {
            "first_name": self.first_name,
            "last_name": self.last_name,
            "can_ride": self.can_ride,
            "type": "User"
        }
    
    def _get_credential_attributes(self):
        return [
            {"name": "first_name", "value": self.first_name},
            {"name": "last_name", "value": self.last_name},
            {"name": "can_ride", "value": str(self.can_ride).lower()}
        ]

In [13]:
class Vertiport(Model):
    def __init__(self, id=None, wallet_name=None, name="", n_airstrip=1, n_parkings=5, coord_lon=0.0, coord_lat=0.0):
        self.name = name
        self.n_airstrip = n_airstrip
        self.n_parkings = n_parkings
        self.coord_lon = coord_lon
        self.coord_lat = coord_lat
        super().__init__(id, wallet_name)
    
    def _get_instance_data(self):
        return {
            "name": self.name,
            "n_airstrip": self.n_airstrip,
            "n_parkings": self.n_parkings,
            "coord_lon": self.coord_lon,
            "coord_lat": self.coord_lat,
            "type": "Vertiport"
        }
    
    def _get_credential_attributes(self):
        return [
            {"name": "last_name", "value": self.name},  # Usamos 'name' como 'last_name' en el schema
            {"name": "n_airstrip", "value": str(self.n_airstrip)},
            {"name": "n_parkings", "value": str(self.n_parkings)},
            {"name": "coord_lon", "value": str(self.coord_lon)},
            {"name": "coord_lat", "value": str(self.coord_lat)}
        ]

In [26]:
issuer.get_entity_credential('evtol')

{'schema_info': {'ver': '1.0',
  'id': 'V4SGRU86Z58d6TV7PBUe6f:2:Evtol_Crede:5.0',
  'name': 'Evtol_Crede',
  'version': '5.0',
  'attrNames': ['status', 'updates', 'id_puerto'],
  'seqNo': 15},
 'definition': 'V4SGRU86Z58d6TV7PBUe6f:3:CL:15:default'}

In [6]:
holder = ACApyClient()

In [9]:
holder.get_existing_cred_defs()

['V4SGRU86Z58d6TV7PBUe6f:3:CL:7:default',
 'V4SGRU86Z58d6TV7PBUe6f:3:CL:9:default',
 'V4SGRU86Z58d6TV7PBUe6f:3:CL:11:default']

In [27]:
te

__main__.ACApyClient